# 02 · Joint iNMF: fitting, choosing k and λ, and the factor atlas

**Model.** Integrative NMF (pyliger, `optimize_ALS`). Each arm *a* (dissociated cells, multiome
nuclei) is factorised as `X_a ≈ H_a (W + V_a)`: `W` is the **shared** gene programme matrix
(k × 3,671 genes), `V_a` a small **arm-private** correction, `H_a` the per-cell usage. pyliger
normalises each cell to its library size and scales every gene by its root-mean-square *within
each arm* before the fit; that per-arm scaling is what removes the modality difference, and λ
penalises the private mass `‖H_a V_a‖²`.

**How k and λ were chosen.** Not by a stability curve (which saturated at ~0.99 for every k in
earlier attempts) but by whether the decomposition **recovers the programmes it was required to
recover**: perilobular, interlobular and medullary identity, inflammatory, mature adipose, ECM
synthesis and myofibroblast/contractile activity. Each fit is scored with a *survival test*
(section 2): a target is *recovered* when one factor is specific to its marker panel, is not
modality-driven, and — for identity programmes — concentrates in the expected annotated state in
every arm that can support it. Fits were run for k = 6–18 at λ = 5 and for λ = 0.1–5 at k = 16.
**k = 16, λ = 5 is the only setting recovering 7 of 7**, with a single modality-driven factor.

**The rest of the notebook** characterises the sixteen factors of that model: the genes each is
made of, which curated panel it matches, where it is used, whether it tracks donor or state,
whether two factors duplicate each other, what the arm-private matrices hold, and all sixteen
on the integrated UMAP. The consolidated annotation table is the input of notebooks 03–05.

Fits are **loaded from disk**; the fit function is included and `FORCE_REFIT` re-runs the grid
into a separate folder without touching the fitted model.

In [1]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')
import hdf5plugin
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

sys.path.insert(0, '/nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/notebooks/revisions/final')
import fb_common as C

C.set_style()
RES, PLT = C.out_dirs('02_inmf_model')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 120)

SIG = C.load_signatures()
PANELS = {**SIG['target'], **SIG['nuisance']}                  # naming panels (section 7): mutually exclusive
TARGET_PANELS = {k: SIG['target'][k] for k in C.SURVIVAL_TARGETS}
SELECTION_PANELS = {**TARGET_PANELS, **SIG['nuisance']}        # the survival test: 7 targets + 3 nuisance panels
light = ad.read_h5ad(C.light_path())
obs = light.obs
print(f'\ncells {int((obs["arm"] == "cells").sum()):,} | nuclei {int((obs["arm"] == "nuclei").sum()):,}')
K_GRID = [6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 18]      # at lambda = 5
LAM_GRID = [0.1, 0.25, 0.5, 1, 2, 5]                    # at k = 16

target     8 panels, 117 genes | id_perilobular, id_interlobular, id_medullary, inflammatory, adipose_mature, ECM_synthesis, Fib_TGFb, myofib_contractile
nuisance   3 panels,  32 genes | IEG_dissoc, proliferation, lymphoid
split      2 panels,  10 genes | adipose_core, adipose_terminal
spatial   15 panels, 117 genes | Adipo_progenitor, Adipo_preadipocyte, Adipo_mature, Adipo_lipidsynth, Lipid_droplet_storage, Lipolysis, FA_uptake_transport, Lipogenesis_denovo, Lipid_master_regulators, Fib_collagen_ECM, Fib_crosslink, Fib_remodel, Fib_EMT_TF, Fib_myofibroblast, Fibrosis_curated_10



cells 70,258 | nuclei 14,858


## 1 · The fit

One (k, λ) per job. `create_liger` requires named obs/var indices, `scale_not_center` subsets each
arm to `var_genes`, and the results land in `varm['W']`, `varm['V']`, `obsm['H']`
(`obsm['H_norm']` after quantile normalisation). `W` is identical across arms and saved once;
`V` and `H` are per arm.

The fit is one function, `_jobs/fit_inmf.py` (printed below so the notebook shows the code it
depends on), submitted one setting per job:

```bash
# one job per (k, lambda); ~6 min and ~20 GB each on 85,116 x 3,671
F=/nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/notebooks/revisions/final
for K in 6 7 8 9 10 11 12 13 14 16 18; do
  bsub -q normal -n 4 -M 32000 -R "select[mem>32000] rusage[mem=32000]" -J inmf_k$K \
       -o $F/_jobs/logs/inmf_k$K.%J.log \
       /nfs/team205/vk8/mambaforge/envs/fb_factors/bin/python $F/_jobs/fit_inmf.py $K 5 \
       /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/data/nmf_joint/final/refit
done
```

In [2]:
import inspect
sys.path.insert(0, f'{C.FINAL}/_jobs')
from fit_inmf import fit_inmf
print(inspect.getsource(fit_inmf))

FORCE_REFIT = False                     # True: re-fit the whole grid in this kernel into data/nmf_joint/final/refit (hours)
REFIT_DIR = f'{C.FINAL_DATA}/refit'


def fit_dir(k, lam):
    """Where a fit lives: the model folder, the k sweep or the lambda sweep archive (or the refit folder)."""
    if FORCE_REFIT:
        return REFIT_DIR
    if (k, lam) == (C.K, C.LAMBDA):
        return C.INMF_DIR
    return C.SWEEP_K_DIR if lam == C.LAMBDA else C.SWEEP_LAM_DIR


def load_fit(k, lam):
    d, tag = fit_dir(k, lam), f'k{k}_lam{lam:g}'
    W = pd.read_csv(f'{d}/W_{tag}.csv', index_col=0)
    H = pd.read_parquet(f'{d}/H_{tag}.parquet')
    V = {a: pd.read_csv(f'{d}/V_{a}_{tag}.csv', index_col=0) for a in C.ARMS}
    H = H.loc[H.index.intersection(obs.index)]
    return W, H, V


if FORCE_REFIT:
    for k in K_GRID:
        fit_inmf(k, C.LAMBDA, REFIT_DIR)
    for lam in LAM_GRID:
        fit_inmf(C.K, lam, REFIT_DIR)

GRID = [(k, C.LAMBDA) for k in K_GRID] + [(C.K, lam) for lam in LAM_GRID if lam != C.LAMBDA]
avail = pd.DataFrame([{'k': k, 'lambda': lam, 'dir': os.path.relpath(fit_dir(k, lam), C.NMF_DIR),
                       'present': os.path.exists(f'{fit_dir(k, lam)}/W_k{k}_lam{lam:g}.csv')} for k, lam in GRID])
print(avail.to_string(index=False))
assert avail['present'].all(), 'a fit is missing -- set FORCE_REFIT = True or check the archive'

def fit_inmf(k, lam, out_dir, input_h5ad=f'{FINAL_DATA}/inmf_input_counts.h5ad',
             basis_txt=f'{FINAL_DATA}/basis_shared.txt', seed=1):
    """pyliger iNMF for one (k, lambda); writes W, V_<arm>, H and H_norm to `out_dir`."""
    import pyliger
    os.makedirs(out_dir, exist_ok=True)
    tag = f'k{k}_lam{lam:g}'
    if os.path.exists(f'{out_dir}/W_{tag}.csv'):
        print(f'{tag}: already fitted in {out_dir}')
        return
    basis = [l.strip() for l in open(basis_txt) if l.strip()]
    adata = ad.read_h5ad(input_h5ad)                       # raw counts, all genes
    arms = []
    for arm in ARMS:
        a = adata[adata.obs['arm'] == arm].copy()
        a.uns['sample_name'] = arm
        a.obs.index.name, a.var.index.name = 'cell', 'gene'
        arms.append(a)
    del adata
    t0 = time.time()
    lig = pyliger.create_liger(arms, remove_missing=False)
    pyliger.normalize(lig)                                 # per-cell library-size normalisation
    lig.var_genes =

## 2 · Scoring a fit: the survival test

For every fit, gene loadings are normalised per factor and z-scored per gene across factors, so a
gene that is high in every factor contributes nothing; a **panel z-score** is the mean of its
genes, z-scored across factors so panels of different sizes compare. Per target programme, the
factor with the highest panel z is examined:

| verdict | rule |
|---|---|
| `weak` | best panel z < 1.5 |
| `modality` | that factor's usage is explained by arm (η² > 0.15) — it encodes assay, not biology |
| `blended` | the runner-up panel is within 0.5 z — the factor carries two programmes |
| `recovered` (identity) | the factor's mean usage is highest in the expected annotated state in every supporting arm, ≥ 2× the runner-up state, with no second state above half-maximum |
| `recovered` (activity) | the factor is panel-specific, arm-balanced and used in every supporting arm — activity programmes are used at graded intensity across states by definition, so single-state concentration is not required |
| `fused` | the same factor is the best factor of two targets — under-factorisation, neither counts |

Concentration is judged on the **original per-arm annotation** (`taa_l4` for cells,
`celltype_v3_L3` for nuclei), which is what existed when the model was chosen. The medullary
programme can only be judged in cells (28 annotated `FB_med` nuclei) and the adipose programme in
nuclei only on its transcribed core (notebook 01). The panel set of the test is the seven target
panels plus the three nuisance panels; `Fib_TGFb` (added to the signature CSV afterwards to name
S1) is not part of the selection criterion and enters only the naming step in section 7.

In [3]:
Z_MIN, GAP_MIN, RATIO_MIN, MODALITY_ETA2 = 1.5, 0.5, 2.0, 0.15


def concentration(usage, labels):
    """Per factor: the state using it most, the ratio to the runner-up state, and how many states sit above half-max."""
    u = C.usage_shares(usage)
    m = u.groupby(np.asarray(labels).astype(str), observed=True).mean()
    out = []
    for f in u.columns:
        s = m[f].sort_values(ascending=False)
        second = s.iloc[1] if len(s) > 1 else np.nan
        out.append(dict(factor=f, top_state=s.index[0], usage=round(float(s.iloc[0]), 4),
                        ratio_2nd=round(float(s.iloc[0] / second), 2) if second and second > 0 else np.inf,
                        n_states_above_half_max=int((s >= s.iloc[0] / 2).sum())))
    return pd.DataFrame(out).set_index('factor')


def arm_diagnostics(usage, arm):
    """Per factor: variance explained by arm and the log2 usage skew between the arms."""
    u = C.usage_shares(usage)
    rows = []
    for f in u.columns:
        means = u[f].groupby(np.asarray(arm), observed=True).mean()
        a, b = means.get('cells', np.nan), means.get('nuclei', np.nan)
        rows.append(dict(factor=f, eta2_arm=round(C.eta_sq(u[f].values, np.asarray(arm)), 4),
                         mean_cells=round(float(a), 4), mean_nuclei=round(float(b), 4),
                         log2_arm_skew=round(float(np.log2((a + 1e-9) / (b + 1e-9))), 2)))
    return pd.DataFrame(rows).set_index('factor')


def survival_test(W, H, obs, target_panels, all_panels, k_label=None):
    """One row per target programme: did this fit recover it?"""
    Z = C.panel_zscores(W, all_panels)
    diag = arm_diagnostics(H, obs['arm'])
    rows = []
    for target in target_panels:
        z = Z.loc[target]
        best, best_z = z.idxmax(), float(z.max())
        other = Z[best].drop(index=target)
        runner_up, gap = other.idxmax(), best_z - float(other.max())
        eta = float(diag.loc[best, 'eta2_arm'])
        support = C.EXPECTED_ARM_SUPPORT[target]
        arms = ['cells'] if support == 'cells_only' else C.ARMS
        conc = {}
        for a in arms:
            exp = C.EXPECTED_TOP_STATE[target].get(a)
            if exp is None:
                continue
            m = (obs['arm'] == a).values
            cc = concentration(H[m], obs.loc[m, 'label'])
            conc[a] = dict(top=cc.loc[best, 'top_state'], ratio=float(cc.loc[best, 'ratio_2nd']),
                           n_half=int(cc.loc[best, 'n_states_above_half_max']), hit=cc.loc[best, 'top_state'] in exp)
        cls = C.PROGRAM_CLASS[target]
        if best_z < Z_MIN:
            verdict = 'weak'
        elif eta > MODALITY_ETA2:
            verdict = 'modality'
        elif gap < GAP_MIN:
            verdict = 'blended'
        elif cls == 'activity':
            used = [float(H.loc[(obs['arm'] == a).values, best].mean()) > 0 for a in arms]
            verdict = 'recovered' if all(used) else 'one_arm_only'
        elif conc and all(c['hit'] for c in conc.values()) and all(c['ratio'] >= RATIO_MIN and c['n_half'] == 1 for c in conc.values()):
            verdict = 'recovered'
        elif conc and any(c['hit'] for c in conc.values()):
            verdict = 'blended'
        else:
            verdict = 'wrong_state'
        row = dict(target=target, klass=cls, factor=best, panel_z=round(best_z, 2), runner_up=runner_up,
                   gap=round(gap, 2), eta2_arm=round(eta, 3), arm_support=support, verdict=verdict)
        for a, cv in conc.items():
            row[f'{a}_top'], row[f'{a}_ratio2nd'], row[f'{a}_n_half'] = cv['top'], round(cv['ratio'], 2), cv['n_half']
        rows.append(row)
    out = pd.DataFrame(rows)
    dupes = out['factor'].duplicated(keep=False)              # one factor claimed by two targets = fused
    out.loc[dupes & out['verdict'].isin(['recovered', 'blended']), 'verdict'] = 'fused'
    out['fused_with'] = out['factor'].map(lambda f: ','.join(sorted(out.loc[out['factor'] == f, 'target'])) if (out['factor'] == f).sum() > 1 else '')
    if k_label is not None:
        out.insert(0, 'k', k_label)
    return out


def summarise_sweep(tables, index='k'):
    df = pd.concat(tables, ignore_index=True)
    grid = df.pivot(index=index, columns='target', values='verdict')
    grid['n_recovered'] = (grid == 'recovered').sum(axis=1)
    return grid


def private_share(W, V):
    """Per factor and arm: fraction of the factor's loading mass sitting in the arm-private V."""
    wm = W.abs().sum(axis=1)
    return pd.DataFrame({a: Vi.abs().sum(axis=1) / (Vi.abs().sum(axis=1) + wm) for a, Vi in V.items()})


def budget_row(W, H, V, obs_):
    """The arm-balance numbers that were bad for cNMF: modality-driven factors, top-3 nuclei share, private share."""
    diag = arm_diagnostics(H, obs_['arm'])
    u = C.usage_shares(H)
    means = u.groupby(obs_['arm'].astype(str).values, observed=True).mean()
    priv = private_share(W, V)
    return dict(n_modality=int((diag['eta2_arm'] > MODALITY_ETA2).sum()), max_eta2_arm=round(float(diag['eta2_arm'].max()), 3),
                max_arm_usage_share=round(float(means.max().max()), 3), top3_nuclei_share=round(float(means.loc['nuclei'].nlargest(3).sum()), 3),
                median_private_cells=round(float(priv['cells'].median()), 4), median_private_nuclei=round(float(priv['nuclei'].median()), 4),
                max_private=round(float(priv.max().max()), 3))

## 3 · The k sweep (λ = 5)

In [4]:
tables, budget = [], []
for k in K_GRID:
    W, H, V = load_fit(k, C.LAMBDA)
    o = obs.loc[H.index]
    tables.append(survival_test(W, H, o, TARGET_PANELS, SELECTION_PANELS, k_label=k))
    budget.append(dict(k=k, **budget_row(W, H, V, o)))
    print(f'k={k:>2}: recovered {int((tables[-1]["verdict"] == "recovered").sum())}/{len(TARGET_PANELS)}', flush=True)
grid_k = summarise_sweep(tables, 'k')
budget_k = pd.DataFrame(budget).set_index('k').join(grid_k['n_recovered'])
grid_k.to_csv(f'{RES}/k_sweep_survival_grid.csv')
pd.concat(tables).to_csv(f'{RES}/k_sweep_survival_detail.csv', index=False)
budget_k.to_csv(f'{RES}/k_sweep_arm_budget.csv')
print('\nverdict per target and k:')
display(grid_k)
print('arm balance per k (n_modality = factors with eta2_arm > 0.15; cNMF k=13 for comparison: 3 modality factors, top-3 nuclei share 0.71):')
display(budget_k)

k= 6: recovered 1/7


k= 7: recovered 2/7


k= 8: recovered 2/7


k= 9: recovered 3/7


k=10: recovered 2/7


...

k=18: recovered 5/7



verdict per target and k:


target,ECM_synthesis,adipose_mature,id_interlobular,id_medullary,id_perilobular,inflammatory,myofib_contractile,n_recovered
k,,,,,,,,
6,weak,blended,blended,fused,fused,recovered,fused,1
7,weak,blended,recovered,blended,fused,recovered,fused,2
8,fused,blended,recovered,blended,fused,recovered,fused,2
9,fused,recovered,recovered,fused,fused,recovered,fused,3
10,recovered,blended,blended,blended,fused,recovered,fused,2
11,fused,blended,blended,blended,recovered,recovered,fused,2
12,fused,recovered,recovered,fused,fused,recovered,fused,3
13,recovered,recovered,recovered,fused,blended,recovered,fused,4
14,fused,recovered,recovered,recovered,fused,recovered,recovered,5


arm balance per k (n_modality = factors with eta2_arm > 0.15; cNMF k=13 for comparison: 3 modality factors, top-3 nuclei share 0.71):


,n_modality,max_eta2_arm,max_arm_usage_share,top3_nuclei_share,median_private_cells,median_private_nuclei,max_private,n_recovered
k,,,,,,,,
6,0,0.056,0.339,0.690,0.0057,0.0388,0.084,1
7,0,0.046,0.330,0.662,0.0038,0.0471,0.087,2
8,0,0.086,0.282,0.598,0.0044,0.0458,0.108,2
9,0,0.060,0.291,0.606,0.0045,0.0543,0.066,3
10,0,0.133,0.249,0.517,0.0054,0.0436,0.094,2
11,1,0.228,0.225,0.492,0.0046,0.0417,0.086,2
12,0,0.065,0.211,0.500,0.0053,0.0417,0.071,3
13,0,0.071,0.276,0.510,0.0042,0.0409,0.089,4
14,1,0.222,0.209,0.477,0.0042,0.0391,0.095,5


In [5]:
fig, axes = plt.subplots(1, 3, figsize=C.figsize_mm('double', 42), layout='constrained')
axes[0].plot(budget_k.index, budget_k['n_recovered'], 'o-', color=C.COLORS['blue'], ms=3)
axes[0].set_ylabel('targets recovered (of 7)'); axes[0].set_ylim(-0.3, 7.5)
axes[1].plot(budget_k.index, budget_k['n_modality'], 'o-', color=C.COLORS['magenta'], ms=3, label='modality-driven factors')
axes[1].plot(budget_k.index, budget_k['max_eta2_arm'] * 10, 's--', color=C.COLORS['orange'], ms=3, label='max η²(arm) × 10')
axes[1].legend(frameon=False); axes[1].set_ylabel('count / scaled η²')
axes[2].plot(budget_k.index, budget_k['top3_nuclei_share'], 'o-', color=C.COLORS['teal'], ms=3)
axes[2].set_ylabel('nuclei usage on their top 3 factors'); axes[2].set_ylim(0, 1)
for ax in axes:
    ax.set_xlabel('k'); ax.axvline(C.K, color='0.7', lw=0.5, ls=':'); ax.grid(False)
    ax.set_xticks(K_GRID)
fig.suptitle('k sweep at λ = 5')
C.save(fig, f'{PLT}/k_sweep.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/k_sweep.pdf   183 x 42 mm


## 4 · The λ sweep (k = 16)

λ penalises the arm-private mass. Lowering it lets each arm keep more of its own signal in `V`
(52 % of the mass at λ = 0.1) and **collapses** recovery — the programmes are genuinely shared,
and more private capacity is strictly worse. At λ = 5 the private matrices carry 0.5 % (cells) and
4 % (nuclei) of the mass.

In [6]:
tables, diag = [], []
for lam in LAM_GRID:
    W, H, V = load_fit(C.K, lam)
    o = obs.loc[H.index]
    t = survival_test(W, H, o, TARGET_PANELS, SELECTION_PANELS, k_label=lam).rename(columns={'k': 'lambda'})
    tables.append(t)
    diag.append(dict(**{'lambda': lam}, **budget_row(W, H, V, o)))
    print(f'lambda={lam:<5}: recovered {int((t["verdict"] == "recovered").sum())}/{len(TARGET_PANELS)}', flush=True)
grid_lam = summarise_sweep(tables, 'lambda')
diag_lam = pd.DataFrame(diag).set_index('lambda').join(grid_lam['n_recovered'])
grid_lam.to_csv(f'{RES}/lambda_sweep_survival_grid.csv')
pd.concat(tables).to_csv(f'{RES}/lambda_sweep_survival_detail.csv', index=False)
diag_lam.to_csv(f'{RES}/lambda_sweep_arm_budget.csv')
display(grid_lam)
display(diag_lam)

lambda=0.1  : recovered 1/7


lambda=0.25 : recovered 5/7


lambda=0.5  : recovered 4/7


lambda=1    : recovered 5/7


lambda=2    : recovered 5/7


lambda=5    : recovered 7/7


target,ECM_synthesis,adipose_mature,id_interlobular,id_medullary,id_perilobular,inflammatory,myofib_contractile,n_recovered
lambda,,,,,,,,
0.10,recovered,blended,blended,fused,blended,blended,fused,1
0.25,recovered,recovered,blended,recovered,blended,recovered,recovered,5
0.50,fused,recovered,recovered,recovered,blended,recovered,fused,4
1.00,recovered,recovered,recovered,blended,blended,recovered,recovered,5
2.00,recovered,recovered,recovered,blended,blended,recovered,recovered,5
5.00,recovered,recovered,recovered,recovered,recovered,recovered,recovered,7


,n_modality,max_eta2_arm,max_arm_usage_share,top3_nuclei_share,median_private_cells,median_private_nuclei,max_private,n_recovered
lambda,,,,,,,,
0.10,0,0.112,0.208,0.445,0.2147,0.5232,0.808,1
0.25,0,0.146,0.215,0.492,0.0633,0.4338,0.830,5
0.50,0,0.143,0.197,0.435,0.0329,0.2430,0.842,4
1.00,1,0.237,0.197,0.457,0.0196,0.1659,0.464,5
2.00,1,0.262,0.182,0.444,0.0101,0.0982,0.214,5
5.00,1,0.176,0.171,0.455,0.0047,0.0418,0.086,7


In [7]:
fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('one-half', 42), layout='constrained')
axes[0].plot(diag_lam.index, diag_lam['n_recovered'], 'o-', color=C.COLORS['blue'], ms=3)
axes[0].set_ylabel('targets recovered (of 7)'); axes[0].set_ylim(-0.3, 7.5)
axes[1].plot(diag_lam.index, diag_lam['median_private_cells'], 'o-', color=C.COLORS['grass'], ms=3, label='cells (median)')
axes[1].plot(diag_lam.index, diag_lam['median_private_nuclei'], 'o-', color=C.GREYS[4], ms=3, label='nuclei (median)')
axes[1].plot(diag_lam.index, diag_lam['max_private'], 's--', color=C.COLORS['orange'], ms=3, label='max over factors')
axes[1].set_ylabel('arm-private share of factor mass'); axes[1].legend(frameon=False)
for ax in axes:
    ax.set_xscale('log'); ax.set_xlabel('λ'); ax.grid(False); ax.set_xticks(LAM_GRID); ax.set_xticklabels([str(l) for l in LAM_GRID])
fig.suptitle('λ sweep at k = 16')
C.save(fig, f'{PLT}/lambda_sweep.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/lambda_sweep.pdf   130 x 42 mm


## 5 · The chosen model: k = 16, λ = 5

In [8]:
W, H, V = load_fit(C.K, C.LAMBDA)
obs = obs.loc[H.index]
light = light[H.index].copy()
U = C.usage_shares(H)
FACTORS = list(W.index)
assert FACTORS == C.FACTORS and (W.values >= 0).all()
basis_final = [l.strip() for l in open(f'{C.FINAL_DATA}/basis_shared.txt') if l.strip()]
assert set(basis_final) == set(W.columns), 'the model was not fitted on the basis of notebook 01'
print(f'W {W.shape} | H {H.shape} | basis matches notebook 01 ({len(basis_final):,} genes)')

detail = survival_test(W, H, obs, TARGET_PANELS, SELECTION_PANELS)
detail.to_csv(f'{RES}/survival_{C.TAG}.csv', index=False)
display(detail)
TARGET_OF = dict(zip(detail['factor'], detail['target']))
CLASS_OF = dict(zip(detail['factor'], detail['klass']))

W (16, 3671) | H (85116, 16) | basis matches notebook 01 (3,671 genes)


,target,klass,factor,panel_z,runner_up,gap,eta2_arm,arm_support,verdict,cells_top,cells_ratio2nd,cells_n_half,nuclei_top,nuclei_ratio2nd,nuclei_n_half,fused_with
0,id_perilobular,identity,S5,3.41,myofib_contractile,3.38,0.032,both,recovered,Fb-perilo_COLEC11,2.36,1,FB_perilo_COLEC11,2.60,1.0,
1,id_interlobular,identity,S9,3.02,lymphoid,3.27,0.010,both,recovered,Fb-interlo_PI16,5.02,1,FB_interlo,2.35,1.0,
2,id_medullary,identity,S14,3.65,id_perilobular,3.36,0.023,cells_only,recovered,Fb-med,4.19,1,NaN,NaN,NaN,
3,adipose_mature,identity,S15,3.74,lymphoid,3.99,0.002,both_core_only,recovered,Fb-adipo,6.66,1,FB_adipo,5.65,1.0,
4,inflammatory,activity,S7,2.71,IEG_dissoc,2.80,0.029,both,recovered,Fb-interm,2.65,1,FB_interm,2.12,1.0,
5,ECM_synthesis,activity,S10,3.56,myofib_contractile,2.78,0.011,both,recovered,Fb-interlo,1.94,2,FB_TGFB1,1.67,2.0,
6,myofib_contractile,activity,S13,3.30,id_medullary,2.85,0.010,both,recovered,Fb-med,3.53,1,FB_med,5.65,1.0,


## 6 · What each factor is made of

Three gene rankings per factor, because no single one is safe:

- **raw** — the genes carrying the most loading mass. Honest, but dominated by highly expressed genes.
- **specific** — loadings z-scored *across the 16 factors*, gated to the factor's top 50 genes by
  mass. Ungated, the z is dominated by unexpressed genes whose across-factor SD is tiny (the lists
  fill with lncRNAs and unannotated loci).
- **distinct** — loading × the fraction of that gene's total loading held by this factor.
  Self-gating; **this is the ranking factors are named from and signatures are built from.**

In [9]:
N_TOP_GENES, MASS_GATE = 20, 50
W_norm, Z_gene = C.gene_zscores(W)
W_dist = C.distinctiveness(W_norm)


def top_specific(f, n=N_TOP_GENES, mass_gate=MASS_GATE):
    z = Z_gene.loc[f]
    if mass_gate:
        z = z.loc[W_norm.loc[f].nlargest(mass_gate).index]
    return list(z.nlargest(n).index)


top_genes = pd.DataFrame({f: {'top_distinct': ', '.join(W_dist.loc[f].nlargest(N_TOP_GENES).index),
                              'top_specific': ', '.join(top_specific(f)),
                              'top_raw': ', '.join(W_norm.loc[f].nlargest(N_TOP_GENES).index)} for f in FACTORS}).T
top_genes.to_csv(f'{RES}/top_genes_{C.TAG}.csv')
for f in FACTORS:
    print(f'{f:4s} {C.FACTOR_NAMES[f]:28s} | ' + ', '.join(W_dist.loc[f].nlargest(12).index))

S1   TGFb / hypoxia / mechano     | JARID2, FLNC, TET3, ZNF469, SERPINB10, SHB, SERPINE1, RGS20, INHBA, BMP6, NEK10, GLIS3
S2   lymphoid (contamination)     | PTPRC, IKZF1, LEF1, BCL11B, RHOH, PRKCB, TMC8, SLAMF1, CD247, CD53, CD3E, SKAP1
S3   interlobular (CCN5)          | CCN5, PTGIS, CLU, FRAS1, C3, RARRES1, DPP4, XG, TNXB, CTSK, LEPR, CHRDL1
S4   proliferation                | TOP2A, UBE2C, TPX2, BIRC5, MKI67, CDK1, CDC20, ANLN, CDCA3, AURKB, CENPF, KNL1
S5   perilobular                  | PTN, COLEC11, ANGPTL1, COL15A1, MYOC, DPT, FGL2, SMOC2, LDB2, MXRA5, NTRK2, KCTD12
S6   IEG / dissociation           | HSPA1B, HSPA1A, DNAJB1, JUN, BAG3, ATF3, HSPH1, HSP90AA1, FOS, EGR1, ZFP36, KLF2
S7   inflammatory                 | PTX3, CTSC, IL1RL1, CTSL, RGCC, FOSL1, FJX1, PAPPA, UPP1, LINC01348, MEDAG, CSF3
S8   perilobular (nuclear)        | ADAMTS9-AS2, PLEKHH2, HAND2-AS1, DACH1, SCN7A, NBEA, PPM1H, FIGN, DOCK5, ADGRL2, RNF150, UNC5C
S9   interlobular (PI16)          | PI16, SEMA3C, KCN

In [10]:
N_BAR = 12
fig, axes = plt.subplots(4, 4, figsize=C.figsize_mm('double', 190), layout='constrained')
for ax, f in zip(axes.ravel(), FACTORS):
    s = W_dist.loc[f].nlargest(N_BAR)[::-1]
    ax.barh(np.arange(N_BAR), s.values, color=C.COLORS['blue'], height=0.7)
    ax.set_yticks(np.arange(N_BAR)); ax.set_yticklabels(s.index)
    ax.set_title(C.FACTOR_LABELS[f]); ax.grid(False); ax.margins(y=0.02)
    ax.tick_params(axis='x', labelsize=4.5)
for ax in axes[-1]:
    ax.set_xlabel('distinctiveness')
C.save(fig, f'{PLT}/top_distinct_genes_{C.TAG}.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/top_distinct_genes_k16_lam5.pdf   183 x 190 mm


## 7 · Which curated panel does each factor match?

Seven target panels plus `Fib_TGFb` (promoted from the Xenium fibrosis sets because it is the
only panel that names S1) and three nuisance panels (IEG/dissociation, proliferation, lymphoid),
so an artefact factor says so instead of being read as biology. A factor is *assigned* a panel
when its best panel z ≥ 1.5 and beats the runner-up by ≥ 0.5; otherwise it stays `unassigned`
and is named by eye from its genes (curated names in `fb_common.FACTOR_NAMES`).

The Xenium adipose/fibrosis gene sets are scored on the same factors separately: they overlap
each other and the identity panels by design, so they never compete for a factor's name.

In [11]:
panel_z = C.panel_zscores(W, PANELS)
panel_z.to_csv(f'{RES}/panel_zscores_{C.TAG}.csv')
best, best_z = panel_z.idxmax(), panel_z.max()
runner = panel_z.apply(lambda col: col.nlargest(2).iloc[1])
calls = pd.DataFrame({'best_panel': best, 'panel_z': best_z.round(2), 'gap': (best_z - runner).round(2)})
calls['assigned'] = (calls['panel_z'] >= Z_MIN) & (calls['gap'] >= GAP_MIN)
display(calls.loc[FACTORS])
print(f'assigned a panel: {int(calls["assigned"].sum())} of {len(calls)}')

order = list(panel_z.max().sort_values(ascending=False).index)
annot = panel_z[order].map(lambda v: f'{v:.1f}' if abs(v) >= 1.5 else '')
fig, ax = plt.subplots(figsize=C.figsize_mm('double', 58), layout='constrained')
im = C.heatmap(ax, panel_z[order], C.CMAP_DIV, -2.5, 2.5, annot=annot, annot_fontsize=4.2)
ax.set_xticklabels([C.FACTOR_LABELS[f] for f in order], rotation=90)
ax.set_title('panel z-scores (panels × factors)')
fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02, label='z across factors')
C.save(fig, f'{PLT}/panel_zscores_{C.TAG}.pdf')
plt.show()

spatial_z = C.panel_zscores(W, SIG['spatial'])
spatial_z.to_csv(f'{RES}/spatial_panel_zscores_{C.TAG}.csv')
spatial_map = pd.DataFrame({'top_factor': spatial_z.idxmax(axis=1), 'z': spatial_z.max(axis=1).round(2),
                            'gap_to_2nd': (spatial_z.max(axis=1) - spatial_z.apply(lambda r: r.nlargest(2).iloc[1], axis=1)).round(2)})
spatial_map['top_factor_name'] = spatial_map['top_factor'].map(C.FACTOR_NAMES)
spatial_map.to_csv(f'{RES}/spatial_set_to_factor_{C.TAG}.csv')
print('\nXenium gene sets -> the factor they point at:')
display(spatial_map.sort_values('z', ascending=False))

,best_panel,panel_z,gap,assigned
S1,Fib_TGFb,2.35,1.97,True
S2,lymphoid,3.75,3.87,True
S3,id_interlobular,1.58,1.34,True
S4,proliferation,3.75,3.81,True
S5,id_perilobular,3.41,3.38,True
S6,IEG_dissoc,3.71,2.75,True
S7,inflammatory,2.71,2.26,True
S8,id_perilobular,0.55,0.80,False
S9,id_interlobular,3.02,2.94,True
S10,ECM_synthesis,3.56,2.53,True


assigned a panel: 13 of 16


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/panel_zscores_k16_lam5.pdf   183 x 58 mm

Xenium gene sets -> the factor they point at:


,top_factor,z,gap_to_2nd,top_factor_name
FA_uptake_transport,S15,3.73,3.74,mature adipose
Adipo_mature,S15,3.72,3.56,mature adipose
Lipid_droplet_storage,S15,3.71,3.51,mature adipose
Adipo_preadipocyte,S15,3.68,3.35,mature adipose
Fibrosis_curated_10,S10,3.46,2.78,ECM synthesis
Lipolysis,S15,3.44,2.49,mature adipose
Lipid_master_regulators,S15,3.03,1.96,mature adipose
Fib_crosslink,S10,2.96,1.94,ECM synthesis
Adipo_lipidsynth,S15,2.60,1.19,mature adipose
Fib_myofibroblast,S10,2.39,0.29,ECM synthesis


## 8 · Where each factor lives, and arm balance

`ratio_2nd` is the top state's mean usage over the runner-up's (concentration; identity
programmes should be concentrated, activity programmes need not be). `eta2_arm` is the fraction
of usage variance explained by assay — above 0.15 the factor encodes cell-vs-nucleus.

In [12]:
where = arm_diagnostics(H, obs['arm'])
for arm in C.ARMS:
    m = (obs['arm'] == arm).values
    where = where.join(concentration(H[m], obs.loc[m, 'label']).add_prefix(f'{arm}_'))
where.to_csv(f'{RES}/where_factors_live_{C.TAG}.csv')
display(where[['eta2_arm', 'mean_cells', 'mean_nuclei', 'log2_arm_skew', 'cells_top_state', 'cells_ratio_2nd', 'nuclei_top_state', 'nuclei_ratio_2nd']])
print('modality-driven (eta2_arm > 0.15):', list(where.index[where['eta2_arm'] > MODALITY_ETA2]))
for arm in C.ARMS:
    top3 = U[(obs['arm'] == arm).values].mean().nlargest(3)
    print(f'{arm:7s} spend {top3.sum():.3f} of usage on their top 3 factors: {", ".join(top3.index)}')

,eta2_arm,mean_cells,mean_nuclei,log2_arm_skew,cells_top_state,cells_ratio_2nd,nuclei_top_state,nuclei_ratio_2nd
factor,,,,,,,,
S1,0.0070,0.0284,0.0466,-0.72,Fb-interlo,1.19,FB_TGFB1,4.02
S2,0.0021,0.0135,0.0196,-0.54,Fb-med,1.35,FB_med,5.01
S3,0.0001,0.0826,0.0795,0.05,Fb-interlo_PI16,1.20,FB_interlo,1.93
S4,0.0008,0.0091,0.0121,-0.40,Fb-interlo,2.50,FB_TGFB1,1.29
S5,0.0319,0.1087,0.0381,1.51,Fb-perilo_COLEC11,2.36,FB_perilo_COLEC11,2.60
S6,0.0023,0.0974,0.0828,0.23,Fb-perilo,1.14,FB_perilo,2.20
S7,0.0294,0.1252,0.0464,1.43,Fb-interm,2.65,FB_interm,2.12
S8,0.0931,0.0630,0.1668,-1.41,Fb-perilo_COLEC11,1.76,FB_perilo_COLEC11,1.42
S9,0.0103,0.0552,0.0885,-0.68,Fb-interlo_PI16,5.02,FB_interlo,2.35


modality-driven (eta2_arm > 0.15): ['S16']
cells   spend 0.361 of usage on their top 3 factors: S11, S7, S5
nuclei  spend 0.455 of usage on their top 3 factors: S16, S8, S11


## 9 · Biology or donor? Covariate association within arm

η² of each factor's usage against state, donor, age group and study, computed separately in each
arm (pooling would let the arm difference leak into every covariate that is unbalanced between
arms). Donor η² is not comparable to state η² (47 donor levels against ~7 states), so the question
that matters — does the factor separate states *inside a single donor*? — is answered by the
within-donor state η², averaged over donors with ≥ 200 cells and ≥ 3 states.

In [13]:
COVARIATES = [c for c in ['label', 'donor_comb', 'age_group', 'study'] if c in obs.columns]
rows = []
for arm in C.ARMS:
    m = (obs['arm'] == arm).values
    for f in FACTORS:
        row = {'factor': f, 'arm': arm}
        for cov in COVARIATES:
            row[cov] = round(C.eta_sq(U.loc[m, f].values, obs.loc[m, cov].astype(str).values), 4)
        rows.append(row)
eta = pd.DataFrame(rows).set_index(['factor', 'arm'])
eta.to_csv(f'{RES}/covariate_eta2_{C.TAG}.csv')


def state_within_donor(arm, min_cells=200, min_states=3):
    m = (obs['arm'] == arm).values
    o, u = obs.loc[m], U.loc[m]
    usable = [(d, i) for d, i in o.groupby('donor_comb', observed=True).groups.items()
              if len(i) >= min_cells and o.loc[i, 'label'].nunique() >= min_states]
    rows = []
    for f in FACTORS:
        e = [C.eta_sq(u.loc[i, f].values, o.loc[i, 'label'].astype(str).values) for _, i in usable]
        e = [x for x in e if np.isfinite(x)]
        rows.append({'factor': f, f'eta2_state_within_donor_{arm}': round(float(np.mean(e)), 4) if e else np.nan, f'n_donors_{arm}': len(e)})
    return pd.DataFrame(rows).set_index('factor')


state_check = pd.concat([eta.xs('cells', level='arm')[['label', 'donor_comb']].rename(columns={'label': 'eta2_state_pooled_cells', 'donor_comb': 'eta2_donor_pooled_cells'}),
                         state_within_donor('cells'), state_within_donor('nuclei')], axis=1).loc[FACTORS]
state_check.to_csv(f'{RES}/state_within_donor_{C.TAG}.csv')
display(state_check.sort_values('eta2_state_within_donor_cells', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('one-half', 70), layout='constrained', sharey=True)
vmax = float(eta.max().max())
for ax, arm in zip(axes, C.ARMS):
    M = eta.xs(arm, level='arm').loc[FACTORS]
    im = C.heatmap(ax, M, 'viridis', 0, vmax, xrot=45, ylabels=(arm == 'cells'), annot=M.map(lambda v: f'{v:.2f}'), annot_fontsize=4.2)
    ax.set_title(C.ARM_NAMES[arm])
    if arm == 'cells':
        ax.set_yticklabels([C.FACTOR_LABELS[f] for f in FACTORS])
fig.colorbar(im, ax=axes, shrink=0.5, pad=0.02, label='η² (variance explained)')
fig.suptitle('variance in factor usage explained by each covariate, within arm')
C.save(fig, f'{PLT}/covariate_eta2_{C.TAG}.pdf')
plt.show()

,eta2_state_pooled_cells,eta2_donor_pooled_cells,eta2_state_within_donor_cells,n_donors_cells,eta2_state_within_donor_nuclei,n_donors_nuclei
factor,,,,,,
S9,0.4279,0.1394,0.4818,25,0.4335,4
S5,0.3048,0.5897,0.3513,25,0.1018,4
S13,0.2682,0.2966,0.3019,25,0.0724,4
S11,0.2600,0.3587,0.2919,25,0.3360,4
S8,0.2740,0.2767,0.2512,25,0.4320,4
S3,0.2775,0.5332,0.2510,25,0.1838,4
S14,0.3293,0.4086,0.2442,25,0.1877,4
S10,0.1492,0.4669,0.1931,25,0.1113,4
S7,0.5556,0.5540,0.1857,25,0.3072,4


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/covariate_eta2_k16_lam5.pdf   130 x 70 mm


## 10 · Redundancy — is any factor the same programme twice?

Usage correlation across cells (do the same cells use both?) and gene-loading cosine on `W` (are
they built from the same genes?). High on both = one programme split in two. S5 and S8 are the one
such pair: the same perilobular identity, split by assay (S8 carries the long, intron-rich
transcripts snRNA-seq retains).

In [14]:
usage_rho = pd.DataFrame(spearmanr(U.fillna(0).values).statistic, index=FACTORS, columns=FACTORS)
W_l2 = W_norm.div(np.sqrt((W_norm ** 2).sum(axis=1)), axis=0)
gene_cos = pd.DataFrame(W_l2.values @ W_l2.values.T, index=FACTORS, columns=FACTORS)
tops = {f: set(top_specific(f, 50)) for f in FACTORS}
pairs = pd.DataFrame([{'a': a, 'b': b, 'usage_rho': round(float(usage_rho.loc[a, b]), 3), 'gene_cosine': round(float(gene_cos.loc[a, b]), 3),
                       'top50_overlap': len(tops[a] & tops[b])} for i, a in enumerate(FACTORS) for b in FACTORS[i + 1:]])
pairs.to_csv(f'{RES}/redundancy_pairs_{C.TAG}.csv', index=False)
print('most gene-similar pairs:'); display(pairs.nlargest(6, 'gene_cosine'))
print('most co-used pairs:'); display(pairs.nlargest(6, 'usage_rho'))

dist = 1 - gene_cos.values.copy(); np.fill_diagonal(dist, 0); dist = (dist + dist.T) / 2
order = [FACTORS[i] for i in leaves_list(linkage(squareform(dist, checks=False), 'average'))]
fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('one-half', 62), layout='constrained')
for ax, (M, title, vlim) in zip(axes, [(usage_rho, 'usage Spearman ρ (all cells)', 0.5), (gene_cos, 'gene-loading cosine (W)', 1.0)]):
    A = M.loc[order, order].copy(); np.fill_diagonal(A.values, np.nan)
    im = C.heatmap(ax, A, C.CMAP_DIV, -vlim, vlim)
    ax.set_xticklabels([C.FACTOR_LABELS[f] for f in order], rotation=90); ax.set_yticklabels([C.FACTOR_LABELS[f] for f in order])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
C.save(fig, f'{PLT}/redundancy_{C.TAG}.pdf')
plt.show()

most gene-similar pairs:


,a,b,usage_rho,gene_cosine,top50_overlap
79,S7,S12,0.259,0.540,11
10,S1,S12,0.121,0.459,3
34,S3,S9,0.294,0.447,5
83,S7,S16,0.065,0.446,4
56,S5,S8,0.152,0.414,2
5,S1,S7,0.060,0.409,4


most co-used pairs:


,a,b,usage_rho,gene_cosine,top50_overlap
62,S5,S14,0.310,0.377,3
34,S3,S9,0.294,0.447,5
114,S13,S14,0.282,0.325,6
79,S7,S12,0.259,0.540,11
94,S9,S12,0.223,0.313,3
61,S5,S13,0.192,0.327,2


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/redundancy_k16_lam5.pdf   130 x 62 mm


## 11 · What the arm-private matrices hold

`private_share = ‖V_a[f]‖₁ / (‖W[f]‖₁ + ‖V_a[f]‖₁)`. Small at λ = 5, but its content is a direct
read-out of what snRNA-seq and scRNA-seq disagree about within a programme.

In [15]:
priv = private_share(W, V)
private = pd.DataFrame([{'factor': f, 'arm': a, 'private_share': round(float(priv.loc[f, a]), 4),
                         'top_private_genes': ', '.join(V[a].loc[f].nlargest(10).index)} for f in FACTORS for a in C.ARMS])
private.to_csv(f'{RES}/private_V_{C.TAG}.csv', index=False)
for a, Vi in V.items():
    print(f'{a:7s} overall private share: {float(Vi.abs().values.sum() / (W.abs().values.sum() + Vi.abs().values.sum())):.4f}')
display(private.sort_values('private_share', ascending=False).head(8))

cells   overall private share: 0.0088
nuclei  overall private share: 0.0424


,factor,arm,private_share,top_private_genes
27,S14,nuclei,0.0857,"DDX58, MX1, IFIT3, MX2, HBB, HBA2, IFIH1, HERC6, CR2, GBP1"
29,S15,nuclei,0.0788,"AOC3, PDE3B, PLIN5, AGT, BMPER, ITGA7, TRHDE, PLXNA4, TLN2, SPECC1"
25,S13,nuclei,0.0730,"ANK3, SGIP1, PCDH1, LINC02408, KCNIP1, SCN3A, NOTCH3, MIR924HG, FHL5, NRXN3"
7,S4,nuclei,0.0636,"TOP2A, KIF18B, TRBC2, CENPF, MKI67, ASPM, CD3G, DIAPH3, KIF4A, DEPDC1"
13,S7,nuclei,0.0469,"LDLRAD3, DEC1, COL4A1, ADAMTS4, COL4A2, FAM124A, SPON1, COL27A1, PTX3, AP000331.1"
9,S5,nuclei,0.0461,"APOD, CXCL14, IGFBP7, MYOC, EDNRB, COL15A1, FBLN5, MGP, GNA14, C7"
11,S6,nuclei,0.0451,"AC011511.2, HSPD1, C1QTNF1-AS1, HSP90AB1, PLIN2, TCF7, SOD2, AL691403.1, ZC3HAV1, HSPE1"
21,S11,nuclei,0.0438,"AP002989.1, AC098829.1, PTPRT, KCND2, TMEM132C, PALM2-AKAP2, PLXNA4, PDGFD, TLN2, ADAMTS12"


## 12 · Consolidated factor annotation

One row per factor: the curated name used on every figure, the data-driven panel call, the
survival-test target, arm balance, top states, within-donor state η² and the leading genes.

In [16]:
annotation = pd.DataFrame({
    'name': pd.Series(C.FACTOR_NAMES), 'panel_call': pd.Series(np.where(calls['assigned'], calls['best_panel'], 'unassigned'), index=calls.index),
    'best_panel': calls['best_panel'], 'panel_z': calls['panel_z'], 'panel_gap': calls['gap'],
    'target': pd.Series(TARGET_OF).reindex(FACTORS).fillna(''), 'program_class': pd.Series(CLASS_OF).reindex(FACTORS).fillna(''),
    'eta2_arm': where['eta2_arm'], 'modality_driven': where['eta2_arm'] > MODALITY_ETA2,
    'mean_usage_cells': where['mean_cells'], 'mean_usage_nuclei': where['mean_nuclei'], 'log2_arm_skew': where['log2_arm_skew'],
    'top_state_cells': where['cells_top_state'], 'ratio_2nd_cells': where['cells_ratio_2nd'],
    'top_state_nuclei': where['nuclei_top_state'], 'ratio_2nd_nuclei': where['nuclei_ratio_2nd'],
    'eta2_state_within_donor_cells': state_check['eta2_state_within_donor_cells'],
    'private_cells': priv['cells'].round(4), 'private_nuclei': priv['nuclei'].round(4),
    'top_genes': top_genes['top_distinct'], 'top_specific': top_genes['top_specific'],
}).loc[FACTORS]
annotation.index.name = 'factor'
annotation.to_csv(f'{RES}/factor_annotation_{C.TAG}.csv')
display(annotation.drop(columns=['top_genes', 'top_specific']))

# the same calls the model was published with?
orig = pd.read_csv(f'{C.INMF_DIR}/factor_annotation_{C.TAG}.csv', index_col=0)
diff = [(f, orig.loc[f, 'best_panel'], annotation.loc[f, 'best_panel']) for f in FACTORS if orig.loc[f, 'best_panel'] != annotation.loc[f, 'best_panel']]
print('best-panel calls differing from the recorded annotation:', diff or 'none')
surv0 = pd.read_csv(f'{C.INMF_DIR}/survival_{C.TAG}.csv')
assert dict(zip(surv0['target'], surv0['factor'])) == dict(zip(detail['target'], detail['factor'])), 'target -> factor mapping changed'
print('OK: the seven target programmes map to the same factors as recorded')

,name,panel_call,best_panel,panel_z,panel_gap,target,program_class,eta2_arm,modality_driven,mean_usage_cells,mean_usage_nuclei,log2_arm_skew,top_state_cells,ratio_2nd_cells,top_state_nuclei,ratio_2nd_nuclei,eta2_state_within_donor_cells,private_cells,private_nuclei
factor,,,,,,,,,,,,,,,,,,,
S1,TGFb / hypoxia / mechano,Fib_TGFb,Fib_TGFb,2.35,1.97,,,0.0070,False,0.0284,0.0466,-0.72,Fb-interlo,1.19,FB_TGFB1,4.02,0.0433,0.0170,0.0310
S2,lymphoid (contamination),lymphoid,lymphoid,3.75,3.87,,,0.0021,False,0.0135,0.0196,-0.54,Fb-med,1.35,FB_med,5.01,0.0169,0.0310,0.0320
S3,interlobular (CCN5),id_interlobular,id_interlobular,1.58,1.34,,,0.0001,False,0.0826,0.0795,0.05,Fb-interlo_PI16,1.20,FB_interlo,1.93,0.2510,0.0040,0.0336
S4,proliferation,proliferation,proliferation,3.75,3.81,,,0.0008,False,0.0091,0.0121,-0.40,Fb-interlo,2.50,FB_TGFB1,1.29,0.0184,0.0033,0.0636
S5,perilobular,id_perilobular,id_perilobular,3.41,3.38,id_perilobular,identity,0.0319,False,0.1087,0.0381,1.51,Fb-perilo_COLEC11,2.36,FB_perilo_COLEC11,2.60,0.3513,0.0004,0.0461
S6,IEG / dissociation,IEG_dissoc,IEG_dissoc,3.71,2.75,,,0.0023,False,0.0974,0.0828,0.23,Fb-perilo,1.14,FB_perilo,2.20,0.1278,0.0054,0.0451
S7,inflammatory,inflammatory,inflammatory,2.71,2.26,inflammatory,activity,0.0294,False,0.1252,0.0464,1.43,Fb-interm,2.65,FB_interm,2.12,0.1857,0.0014,0.0469
S8,perilobular (nuclear),unassigned,id_perilobular,0.55,0.80,,,0.0931,False,0.0630,0.1668,-1.41,Fb-perilo_COLEC11,1.76,FB_perilo_COLEC11,1.42,0.2512,0.0180,0.0164
S9,interlobular (PI16),id_interlobular,id_interlobular,3.02,2.94,id_interlobular,identity,0.0103,False,0.0552,0.0885,-0.68,Fb-interlo_PI16,5.02,FB_interlo,2.35,0.4818,0.0058,0.0353


best-panel calls differing from the recorded annotation: none
OK: the seven target programmes map to the same factors as recorded


## 13 · All sixteen factors on the integrated UMAP

Colour is the factor's **share** of a cell's usage, each panel scaled 0 – its 99th percentile.
The second figure splits the arms: a programme claimed as shared must show structure in both.

In [17]:
xy = np.asarray(light.obsm['X_umap'])
arms = obs['arm'].astype(str).values
Ua = U.reindex(light.obs_names)
fig = C.umap_factor_grid(xy, arms, Ua, C.FACTOR_LABELS, FACTORS, ncols=4, panel_height_mm=42)
C.save(fig, f'{PLT}/umap_factors_{C.TAG}.pdf')
plt.show()
for arm in C.ARMS:
    fig = C.umap_factor_grid(xy, arms, Ua, C.FACTOR_LABELS, FACTORS, arm=arm, ncols=4, panel_height_mm=42,
                             point_size=0.35 if arm == 'cells' else 0.6)
    fig.suptitle(f'{C.ARM_NAMES[arm]} coloured, the other arm in grey', fontsize=plt.rcParams['axes.titlesize'])
    C.save(fig, f'{PLT}/umap_factors_{arm}_{C.TAG}.pdf', close=True)

fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('double', 70), layout='constrained')
C.umap_categorical(xy, arms, C.ARM_COLORS, order=C.ARMS, title='modality', ax=axes[0])
C.umap_categorical(xy, obs['label'].astype(str).values, {**{l: C.COLORS['blue'] for l in ['Fb-perilo', 'Fb-perilo_COLEC11', 'FB_perilo', 'FB_perilo_COLEC11']},
                                                          **{l: C.COLORS['orange'] for l in ['Fb-interlo', 'Fb-interlo_PI16', 'FB_interlo']},
                                                          'Fb-interm': C.COLORS['teal'], 'FB_interm': C.COLORS['teal'], 'Fb-med': C.COLORS['magenta'], 'FB_med': C.COLORS['magenta'],
                                                          'Fb-adipo': C.COLORS['yellow'], 'FB_adipo': C.COLORS['yellow'], 'FB_TGFB1': C.COLORS['purple']},
                   title='original per-arm annotation (model selection)', ax=axes[1], legend_ncol=2)
C.save(fig, f'{PLT}/umap_context_{C.TAG}.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/umap_factors_k16_lam5.pdf   183 x 172 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/umap_factors_cells_k16_lam5.pdf   183 x 172 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/umap_factors_nuclei_k16_lam5.pdf   183 x 172 mm


-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/02_inmf_model/umap_context_k16_lam5.pdf   183 x 70 mm


## Files written

| file | contents |
|---|---|
| `results/final/02_inmf_model/k_sweep_*.csv`, `lambda_sweep_*.csv` | survival grids, per-target detail, arm-balance diagnostics |
| `results/final/02_inmf_model/survival_k16_lam5.csv` | the seven targets in the chosen model |
| `results/final/02_inmf_model/top_genes_k16_lam5.csv` | distinct / specific / raw gene rankings per factor |
| `results/final/02_inmf_model/panel_zscores_k16_lam5.csv`, `spatial_*` | panels × factors |
| `results/final/02_inmf_model/where_factors_live_*.csv`, `covariate_eta2_*.csv`, `state_within_donor_*.csv`, `redundancy_pairs_*.csv`, `private_V_*.csv` | diagnostics |
| **`results/final/02_inmf_model/factor_annotation_k16_lam5.csv`** | the consolidated table used by notebooks 03–05 |
| `plots/final/02_inmf_model/*.pdf` | k / λ sweeps, top genes, panel heatmap, η², redundancy, UMAPs |

The model matrices themselves stay where they were fitted: `data/nmf_joint/inmf/{W,V_cells,V_nuclei}_k16_lam5.csv`, `H_k16_lam5.parquet`.